# Video JEPA Training on Moving MNIST - VICReg version

This notebook demonstrates training an Video JEPA (Joint Embedding Predictive Architecture) model on Moving MNIST using the eb_jepa framework.

## Hardware Requirements

- This notebook is designed to run on Google Colab with a GPU runtime
- Recommended: GPU with at least 12GB VRAM (e.g., Tesla T4, P100)
- Training time: ~1-2 hours depending on GPU

## Features

- Self-supervised representation learning using JEPA + VCReg loss
- Optimized configuration for Colab environment
- Optional Weights & Biases logging integration


In [2]:
# Colab setup cell for cloning a private GitHub repo and installing dependencies
# Instructions:
# 1. Generate a GitHub personal access token (PAT) with repo access: https://github.com/settings/tokens
# 2. Paste your token below (or use getpass for more security)
# 3. Replace YOUR_USERNAME and YOUR_REPO with your GitHub username and repo name

# from getpass import getpass
from google.colab import userdata
# Enter your GitHub token securely
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') # getpass('Enter your GitHub personal access token: ')
GITHUB_USER = 'kabairobert'  # <-- change this
REPO_NAME = 'eb_jepa_private'        # <-- change this

In [3]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
print(f'CWD: {os.getcwd()}')
print('Directory contents:', os.listdir())

Mounted at /content/drive
CWD: /content
Directory contents: ['.config', 'drive', 'sample_data']


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

In [4]:
# Check if we're running on GPU
!nvidia-smi

Fri Feb 27 10:09:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# 1. Install uv (standalone, pip-independent)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Add uv's install location to PATH for this session so `!uv` works everywhere
import os
os.environ["PATH"] += ":/root/.cargo/bin"

# 3. Confirm uv is on PATH (should print current uv version)
!uv --version

downloading uv 0.10.6 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.10.6


In [6]:
# 4. Clone the repository and enter it
# Clone the private repo
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}
# Install required packages
# !pip install fire omegaconf wandb tqdm
# !git clone https://github.com/facebookresearch/eb_jepa.git
# %cd eb_jepa
# !pip install -e .

# 5. Sync environment
!uv sync

Cloning into 'eb_jepa_private'...
remote: Enumerating objects: 490, done.
remote: Counting objects: 100% (490/490), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 490 (delta 239), reused 451 (delta 200), pack-reused 0 (from 0)
Receiving objects: 100% (490/490), 8.35 MiB | 15.22 MiB/s, done.
Resolving deltas: 100% (239/239), done.
/content/eb_jepa_private
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 124 packages in 1.69s
Prepared 122 packages in 1m 06s
Installed 122 packages in 1.05s
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.12.1
 + autoflake==2.3.3
 + black==26.1.0
 + bokeh==3.8.2
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + cloudpickle==3.1.2
 + contourpy==1.3.3
 + cycler==0.12.1
 + decorator==5.2.1
 + decord==0.6.0
 + eb-jepa==0.1.1 (from file:///content/eb_jepa_private)
 + einops==0.8.2
 + farama-notifications=

In [7]:
# Set environment variables for datasets and checkpoints
# %env EBJEPA_DSETS=/content/{REPO_NAME}/eb_jepa/datasets
%env EBJEPA_DSETS=/content/drive/MyDrive/Colab Notebooks/JEPA/datasets
# %env EBJEPA_CKPTS=/content/eb_jepa/checkpoints

env: EBJEPA_DSETS=/content/drive/MyDrive/Colab Notebooks/JEPA/datasets


In [8]:
# Create a modified config for Colab
%%writefile examples/video_jepa/cfgs/colab_config_vicreg.yaml
meta:
  seed: 2025
  device: auto  # auto, cuda, or cpu
  # checkpoint_dir: /content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/ # Directory to save checkpoints and logs
  load_model: true # need to set this with the model_folder to actually load the checkpoint!
  model_folder: /content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/video_jepa/dev_2026-02-26_19-07/resnet_bs24_lr0.001_std10.0_cov100.0_seed2025 # just for continuing a stopped train
  load_checkpoint: latest.pth.tar  # or epoch_10.pth.tar for a specific epoch

data:
  dataset: moving_mnist
  batch_size: 24
  num_workers: 2

model:
  # Encoder (ResNet5)
  dobs: 1           # Input channels (grayscale)
  henc: 32          # Hidden dimension in encoder
  dstc: 16          # Output representation dimension

  # Predictor (ResUNet)
  hpre: 32          # Hidden dimension in predictor

  # Training
  steps: 2          # Number of prediction steps during training

loss:
  # Variance-Covariance regularization
  cov_coeff: 100.0  # Covariance loss weight
  std_coeff: 10.0   # Standard deviation loss weight

optim:
  epochs: 50
  lr: 1.0e-3

logging:
  log_wandb: true  # Disable wandb by default
  log_every: 1
  save_every: 10
  vis_every: 0        # Run visualization loop every N epochs during training (0 = disabled in loop)
  vis_final: true     # Run visualization loop after training completes
  tqdm_silent: false

training:
  use_amp: true
  dtype: float16

Writing examples/video_jepa/cfgs/colab_config_vicreg.yaml


In [10]:
# Optional: Configure W&B logging
import wandb
# wandb.login()  # Uncomment to use W&B logging
import os
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
# Run training
!uv run python -m examples.video_jepa.main --fname examples/video_jepa/cfgs/colab_config_vicreg.yaml

Uninstalled 1 package in 3ms
Installed 1 package in 3ms
[INFO    ][2026-02-27 10:11:54][eb_jepa.training_utils][load_config              ] Loaded config from examples/video_jepa/cfgs/colab_config_vicreg.yaml
[INFO    ][2026-02-27 10:11:54][eb_jepa.training_utils][setup_device             ] Using device: cuda
[INFO    ][2026-02-27 10:11:54][eb_jepa.training_utils][setup_seed               ] Random seed set to 2025
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: robertkabai (robertkabai-um) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/video_jepa/dev_2026-02-26_19-07/resnet_bs24_lr0.001_std10.0_cov100.0_seed2025/wandb/run-20260227_101157-7bylq0qc
wandb: Run `wandb offline` to turn off sy

## Visualize results

In [ ]:
# Load and visualize results
import torch
import matplotlib.pyplot as plt
from pathlib import Path

def plot_training_progress(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    print(f"Epoch: {checkpoint['epoch']}")
    print(f"Linear Probe Validation Accuracy: {checkpoint.get('linear_val_acc', 'N/A')}%")

    # Add more visualization as needed

# Find the latest checkpoint
latest_checkpoint = Path('/content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/image_jepa/dev_2026-02-22_20-52/resnet_bcs_proj_bs256_ep50_ph2048_po128_lmbd10.0_seed42/latest.pth.tar')
if latest_checkpoint.exists():
    plot_training_progress(latest_checkpoint)
else:
    print("No checkpoint found")

In [ ]:
# Plot accuracy at each checkpoint
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import re

checkpoint_dir = Path('/content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/image_jepa/dev_2026-02-22_20-52/resnet_bcs_proj_bs256_ep50_ph2048_po128_lmbd10.0_seed42')

epochs = []
accuracies = []

if checkpoint_dir.exists() and checkpoint_dir.is_dir():
    # Modified to look for all .pth.tar files
    for checkpoint_file in sorted(checkpoint_dir.glob('*.pth.tar')):
        try:
            checkpoint = torch.load(checkpoint_file, map_location='cpu')
            epoch = checkpoint['epoch']
            # Assuming 'linear_val_acc' is the key for validation accuracy
            accuracy = checkpoint.get('linear_val_acc')
            if accuracy is not None:
                epochs.append(epoch)
                accuracies.append(accuracy)
        except Exception as e:
            print(f"Error loading {checkpoint_file}: {e}")

    if epochs and accuracies:
        # Sort the collected data by epoch to ensure correct plotting order
        sorted_data = sorted(zip(epochs, accuracies), key=lambda x: x[0])
        sorted_epochs, sorted_accuracies = zip(*sorted_data)

        plt.figure(figsize=(10, 6))
        plt.plot(sorted_epochs, sorted_accuracies, marker='o', linestyle='-')
        plt.title('Linear Probe Validation Accuracy per Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Validation Accuracy (%)')
        plt.grid(True)
        plt.xticks(list(sorted_epochs)) # Ensure all epoch numbers are shown if few epochs
        plt.tight_layout()
        plt.show()
    else:
        print("No valid accuracy data found in checkpoints.")
else:
    print(f"Checkpoint directory not found: {checkpoint_dir}")

In [ ]:
# Visualize latest checkpoint
from examples.image_jepa.vis import visualize_from_checkpoint
figs = visualize_from_checkpoint(ckpt_path="/content/drive/MyDrive/Colab Notebooks/JEPA/checkpoints/image_jepa/dev_2026-02-22_20-52/resnet_bcs_proj_bs256_ep50_ph2048_po128_lmbd10.0_seed42/latest.pth.tar", tsne_method="umap")
# figures are saved to disk automatically